In [1]:
%config Completer.use_jedi = False

In [2]:
# load module
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
import copy
import warnings
import torch
import optuna
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger


from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
#from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

# training error 일때만 풀기
#import tensorflow as tf 
#import tensorboard as tb 
#tf.io.gfile = tb.compat.tensorflow_stub.io.gfile

import matplotlib
matplotlib.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")  # avoid printing out absolute paths
plt.rcParams['font.family'] = 'NanumGothic'
#plt.rcParams['font.sans-serif'] = ['NanumGothic.ttf', 'sans-serif']


/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.23ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(
/home/nplab/.local/lib/python3.8/site-packages/pkg_resources/__init__.py:116: PkgResourcesDeprecationWarning: 0.1.36ubuntu1 is an invalid version and will not be supported in a future release
  warnings.warn(


In [3]:
# functions
def ewma_data_processing(path = '../training_data/ewma_6h_scaling.csv'):
    data = pd.read_csv(path)
    data.drop('Unnamed: 0.1'   , axis=1 , inplace=True)
    data.drop('Unnamed: 0'   , axis=1 , inplace=True)
    data['REG_DTIME'] = pd.to_datetime(data['REG_DTIME'])
    data['DOW'] = data['REG_DTIME'].dt.dayofweek
    data['HOD'] = data['REG_DTIME'].dt.hour
    data["time_idx"] =  \
    (data["REG_DTIME"].dt.month) * data["REG_DTIME"].dt.daysinmonth * 24  + \
    data["REG_DTIME"].dt.day * 24  + \
    data["REG_DTIME"].dt.hour 
    data["time_idx"] -= data["time_idx"].min()
    data['h_dong'] = data['h_dong'].astype(str)
    data['DOW'] = data['DOW'].astype(str)
    data['HOD'] = data['HOD'].astype(str)
    data['precip_form'] = data['precip_form'].astype(str)
    data['isHoliday'] = data['isHoliday'].astype(str)

    return data

def org_data_processing(path = '../sample_table/total_data.csv'):
    data = pd.read_csv(path)
    data.drop('Unnamed: 0'   , axis=1 , inplace=True)
    data['REG_DTIME'] = pd.to_datetime(data['REG_DTIME'])
    data['DOW'] = data['REG_DTIME'].dt.dayofweek
    data['HOD'] = data['REG_DTIME'].dt.hour
    data["time_idx"] =  \
    (data["REG_DTIME"].dt.month) * data["REG_DTIME"].dt.daysinmonth * 24  + \
    data["REG_DTIME"].dt.day * 24  + \
    data["REG_DTIME"].dt.hour 
    data["time_idx"] -= data["time_idx"].min()
    data['h_dong'] = data['h_dong'].astype(str)
    data['DOW'] = data['DOW'].astype(str)
    data['HOD'] = data['HOD'].astype(str)
    data['precip_form'] = data['precip_form'].astype(str)
    data['isHoliday'] = data['isHoliday'].astype(str)

    return data

def get_training(data , max_prediction_length, max_encoder_length):
    # traing data 생성
    max_prediction_length = max_prediction_length
    max_encoder_length = max_encoder_length
    training_cutoff = data["time_idx"].max() - max_prediction_length

    training = TimeSeriesDataSet(
        data[lambda x : x.time_idx <= training_cutoff],
        time_idx = "time_idx",
        target = "count",
        group_ids = ['h_dong'],
        min_encoder_length=max_encoder_length // 2,  # keep encoder length long (as it is in the validation set)
        max_encoder_length=max_encoder_length,
        min_prediction_length=1,
        max_prediction_length=max_prediction_length,
        static_categoricals=["h_dong"],
        time_varying_known_categoricals=["HOD", "DOW" , 'isHoliday'],
        time_varying_known_reals=['pops'],
        #variable_groups={"special_days": special_days},  # group of categorical variables can be treated as one variable
        time_varying_unknown_categoricals=['precip_form'],
        time_varying_unknown_reals=['count','windspd' , 'temp' ,'precip'],
        add_relative_time_idx=True,
        add_target_scales=True,
        add_encoder_length=True,
        #allow_missing=True,
        allow_missing_timesteps = True)
    return training

def truncate(num, rat):
    if (num - int(num)) < rat:
        return int(num)
    else:
        return int(num)+1
    
# boolean으로 계산시에는 call하는 함수에서 categoize
def perf_measure_boolean(y_actual, y_pred):
    # y_actual value
    # y_pred  array
    # numberical measuer
    TP =  FP = TN = FN = 0
    
    if y_actual == 1 and sum(y_pred) > 0:
        TP += 1
    if y_actual == 1 and sum(y_pred) == 0:
        FN += 1
    if y_actual == 0 and sum(y_pred) > 0:
        FP += 1
    if y_actual == 0 and sum(y_pred) == 0:
        TN += 1

    return TP, FN ,FP, TN

def perf_measure_numerial(y_actual, y_pred):
    # y_actual one array
    # y_pred  array
    # numberical measuer : set class로 and 연산
    TP =  FP = TN = FN = 0
    inter = np.intersect1d(y_actual , y_pred)

    if len(y_pred) == 1:
        if y_actual[0] == y_pred[0] and y_actual[0] > 0:
            TP += 1
        if y_actual[0] > 0 and y_actual[0] != y_pred[0]:
            FN += 1
        if y_actual[0] == 0 and y_pred[0] > 0 :
            FP += 1
        if y_actual[0] == y_pred[0] == 0:
            TN += 1

    else:
        if len(inter) == 0:
            if y_actual[0] > 0:
                FN +=1
            if y_actual[0] == 0 and sum(y_pred) > 0:
                FP += 1

                
        if len(inter) == 1:
            if y_actual[0] > 0:
                TP +=1
            if y_actual[0] == 0 and sum(y_pred) > 0:
                FP += 1
            elif y_actual[0] == 0:
                TN += 1
    return TP, FN ,FP, TN 

def boolean(num):
    if num == 0:
        return 0
    else:
        return 1

In [28]:
   
# class : load & training
class TFTmodel():
    def __init__(self, org_data , ewma_data , max_prediction_length, max_encoder_length, bs1, bs2):
        #   data preprocessing
        self.ewma_data = ewma_data_processing(ewma_data)
        self.org_data = org_data_processing(org_data)
        self.org_training = get_training(self.org_data,max_prediction_length, max_encoder_length)
        self.ewma_training = get_training(self.ewma_data,max_prediction_length, max_encoder_length)
        
        batch_size= bs1
        self.org_dataloader = self.org_training.to_dataloader(bs1)
        self.org_validation = TimeSeriesDataSet.from_dataset(self.org_training, self.org_data, predict=True, stop_randomization=True)
        batch_size = bs2  # set this between 32 to 128
        self.org_train_dataloader = self.org_training.to_dataloader(train=True, batch_size=bs2, num_workers=0)
        self.org_val_dataloader = self.org_validation.to_dataloader(train=False, batch_size=bs2*100, num_workers=0)
        
        batch_size= bs1
        self.ewma_dataloader = self.ewma_training.to_dataloader(bs1)
        self.ewma_validation = TimeSeriesDataSet.from_dataset(self.ewma_training, self.ewma_data, predict=True, stop_randomization=True)
        batch_size = bs2  # set this between 32 to 128
        self.ewma_train_dataloader = self.ewma_training.to_dataloader(train=True, batch_size=bs2, num_workers=0)
        self.ewma_val_dataloader = self.ewma_validation.to_dataloader(train=False, batch_size=bs2*100, num_workers=0)
        self.dong_scaling_coff = {}
        for dong in self.org_data['h_dong'].unique():
            self.dong_scaling_coff[dong] = 1
        self.metric_mode = None
    
    def print_hparams(self):
        print(self.tft.hparams)
    
        
    def load_model(self, model_path):
        self.load_model = TemporalFusionTransformer.load_from_checkpoint(model_path)

    def pred_plot(self,q):
        pred ,x ,dong_idx = self.load_model.predict(ewma_val_dataloader, return_index = True, mode="raw", return_x = True)
        #plt.plot((pred['prediction'][0,:,q] - pred['prediction'][0,:,q].min()))
        #plt.show()
        


In [29]:
org_data   = '../../training_data/total_data.csv'
ewma_data  = '../../training_data/ewma_6h_scaling.csv'
start = '2021/12/2'
end = '2021/12/3'
model_path = 'test/lightning_logs/version_1/checkpoints/epoch=299-step=1500.ckpt'
window_size = 0

In [30]:
TFTclass = TFTmodel(org_data, ewma_data, 24*30 , 24*30*3, 4, 16)
TFTclass.load_model(model_path)

In [31]:
TFTclass.pred_plot(3)

NameError: name 'ewma_val_dataloader' is not defined